In [3]:
!pip install -r requirements.txt

!playwright install
!playwright install-deps

Playwright Host validation warning: 
╔══════════════════════════════════════════════════════╗
║ Host system is missing dependencies to run browsers. ║
║ Please install them with the following command:      ║
║                                                      ║
║     playwright install-deps                          ║
║                                                      ║
║ Alternatively, use apt:                              ║
║     apt-get install libxcomposite1\                  ║
║         libgtk-3-0\                                  ║
║         libatk1.0-0                                  ║
║                                                      ║
║ <3 Playwright Team                                   ║
╚══════════════════════════════════════════════════════╝
    at validateDependenciesLinux (/usr/local/lib/python3.12/dist-packages/playwright/driver/package/lib/coreBundle.js:27798:9)
    at async Registry._validateHostRequirements (/usr/local/lib/python3.12/dist-packages/playwr

Step 2: Create FastAPI Backend

backend/main.py

In [5]:
import os
import sys

# Ensure the project root is in sys.path for module discovery
# Assuming the 'backend' directory is directly under /content/
project_root = os.getcwd() # This is usually /content/ in Colab
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Create dummy files and directories to satisfy the import for now
# This part implicitly creates the backend/api/routes.py file
# This is a prerequisite for the import to succeed.
backend_dir = os.path.join(project_root, "backend")
api_dir = os.path.join(backend_dir, "api")

os.makedirs(api_dir, exist_ok=True)

init_file_backend = os.path.join(backend_dir, "__init__.py")
if not os.path.exists(init_file_backend):
    with open(init_file_backend, "w") as f:
        pass

init_file_api = os.path.join(api_dir, "__init__.py")
if not os.path.exists(init_file_api):
    with open(init_file_api, "w") as f:
        pass

routes_file = os.path.join(api_dir, "routes.py")
if not os.path.exists(routes_file):
    # Add a minimal content to routes.py so it's a valid module
    with open(routes_file, "w") as f:
        f.write("from fastapi import APIRouter\n\nrouter = APIRouter()")

from fastapi import FastAPI
from backend.api.routes import router

app = FastAPI(
    title="Agentic Web"
)

app.include_router(router)

@app.get("/")
def home():
    return {
        "message":"Agentic Web Running"
    }

Step 3: User Query API

backend/api/routes.py

In [6]:
from fastapi import APIRouter
from pydantic import BaseModel

router = APIRouter()

class UserRequest(BaseModel):
    task:str

@router.post("/execute")
def execute(request:UserRequest):

    return {
        "task":request.task,
        "status":"received"
    }

Step 4: Intent Understanding Agent


backend/agents/intent_agent.py

In [9]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key is None:
    print("\n==========================================================================================")
    print("WARNING: OPENAI_API_KEY environment variable not found.")
    print("Please ensure you have a .env file in your working directory (/content/) with the content:")
    print("OPENAI_API_KEY=YOUR_SECRET_KEY")
    print("Replace 'YOUR_SECRET_KEY' with your actual OpenAI API key.")
    print("Alternatively, you can set it directly in a Colab cell using: os.environ['OPENAI_API_KEY'] = 'YOUR_SECRET_KEY'")
    print("==========================================================================================\n")
    raise ValueError("OPENAI_API_KEY not set. Please provide your OpenAI API key to proceed.")

# Initialize OpenAI client with API key from environment variables
client = OpenAI(api_key=openai_api_key)

def understand_intent(user_task):

    prompt = f"""
    Extract:
    1. Goal
    2. Entities
    3. Constraints

    Task:
    {user_task}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role":"user","content":prompt}
        ]
    )

    return response.choices[0].message.content


Please ensure you have a .env file in your working directory (/content/) with the content:
OPENAI_API_KEY=YOUR_SECRET_KEY
Replace 'YOUR_SECRET_KEY' with your actual OpenAI API key.
Alternatively, you can set it directly in a Colab cell using: os.environ['OPENAI_API_KEY'] = 'YOUR_SECRET_KEY'



ValueError: OPENAI_API_KEY not set. Please provide your OpenAI API key to proceed.

Step 5: Planning Agent

backend/agents/planner.py

In [ ]:
def create_plan(task):

    plan = [
        "Search Website",
        "Collect Data",
        "Compare Options",
        "Execute Action",
        "Verify Result"
    ]

    return plan

Step 6: Browser Automation Agent

backend/browser/browser_agent.py

In [10]:
from playwright.sync_api import sync_playwright

def open_website(url):

    with sync_playwright() as p:

        browser = p.chromium.launch(
            headless=False
        )

        page = browser.new_page()

        page.goto(url)

        title = page.title()

        browser.close()

        return title

In [11]:
print(open_website("https://www.amazon.in"))

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

Step 7: Research Agent

backend/agents/research_agent.py

In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_data(url):

    page = requests.get(url)

    soup = BeautifulSoup(
        page.text,
        "html.parser"
    )

    return soup.title.text

Step 8: Decision Agent

backend/agents/decision_agent.py

In [ ]:
def select_best_option(options):

    best = sorted(
        options,
        key=lambda x:x["price"]
    )

    return best[0]

Example:

In [ ]:
options = [
    {"name":"A","price":5000},
    {"name":"B","price":4000},
    {"name":"C","price":6000}
]

print(select_best_option(options))

Step 9: Memory Module

backend/memory/user_memory.py

In [ ]:
user_preferences = {}

def save_preference(
    user,
    key,
    value
):

    if user not in user_preferences:
        user_preferences[user] = {}

    user_preferences[user][key] = value


def get_preferences(user):

    return user_preferences.get(
        user,
        {}
    )

Step 10: Multi-Agent Orchestrator

backend/agents/orchestrator.py

In [ ]:
from agents.intent_agent import understand_intent
from agents.planner import create_plan

def execute_task(task):

    intent = understand_intent(task)

    plan = create_plan(task)

    return {
        "intent":intent,
        "plan":plan
    }

Step 11: Connect API to Orchestrator

backend/api/routes.py

In [ ]:
from fastapi import APIRouter
from pydantic import BaseModel

from agents.orchestrator import execute_task

router = APIRouter()

class UserRequest(BaseModel):
    task:str

@router.post("/execute")

def execute(request:UserRequest):

    result = execute_task(
        request.task
    )

    return result

Step 13: Self-Healing Recovery Agent (Novel Feature)

In [ ]:
def recover_from_failure(error):

    recovery_map = {
        "button_not_found":
            "search_similar_button",

        "page_timeout":
            "reload_page",

        "network_error":
            "retry_connection"
    }

    return recovery_map.get(
        error,
        "manual_review"
    )

Step 14: Fraud Detection Agent

In [ ]:
def fraud_score(
    ssl,
    reviews,
    domain_age
):

    score = 0

    if ssl:
        score += 30

    if reviews > 100:
        score += 30

    if domain_age > 365:
        score += 40

    return score

Step 15: Run Complete Workflow

In [ ]:
from agents.orchestrator import execute_task

task = """
Book cheapest flight
Delhi to Mumbai
"""

result = execute_task(task)

print(result)

Additional USP

Negotiation Agent

agents/negotiation_agent.py

In [ ]:
class NegotiationAgent:

    def find_best_price(self, products):

        products = sorted(
            products,
            key=lambda x: x["price"]
        )

        best_option = products[0]

        return {
            "selected_product": best_option,
            "savings":
                max([p["price"] for p in products])
                - best_option["price"]
        }

In [ ]:
products = [
    {"site":"A","price":5000},
    {"site":"B","price":4200},
    {"site":"C","price":4600}
]

Fraud Detection Agent

agents/fraud_detection_agent.py

In [ ]:
class FraudDetectionAgent:

    def calculate_score(
        self,
        ssl,
        reviews,
        domain_age
    ):

        score = 0

        if ssl:
            score += 30

        if reviews > 100:
            score += 30

        if domain_age > 365:
            score += 40

        return score

    def is_safe(self, score):

        return score >= 70

Sustainability Agent

agents/sustainability_agent.py

In [ ]:
class SustainabilityAgent:

    def carbon_score(
        self,
        shipping_distance,
        packaging_score
    ):

        return (
            shipping_distance * 0.1
            +
            packaging_score
        )

    def recommend(
        self,
        options
    ):

        return min(
            options,
            key=lambda x:x["carbon_score"]
        )

Self-Healing Agent V2

agents/self_healing_agent.py

In [ ]:
class SelfHealingAgent:

    def recover(
        self,
        error_type
    ):

        recovery_actions = {

            "timeout":
                "refresh_page",

            "button_missing":
                "search_alternative_button",

            "network_failure":
                "retry_connection"
        }

        return recovery_actions.get(
            error_type,
            "manual_review"
        )

Memory Agent

agents/memory_agent.py

In [ ]:
class MemoryAgent:

    def __init__(self):

        self.memory = {}

    def save_preference(
        self,
        user,
        key,
        value
    ):

        if user not in self.memory:
            self.memory[user] = {}

        self.memory[user][key] = value

    def get_preferences(
        self,
        user
    ):

        return self.memory.get(
            user,
            {}
        )

Analytics Agent

agents/analytics_agent.py

In [ ]:
class AnalyticsAgent:

    def task_metrics(
        self,
        total_tasks,
        success_tasks
    ):

        success_rate = (
            success_tasks
            /
            total_tasks
        ) * 100

        return {
            "success_rate":
                round(success_rate,2)
        }

Recommendation Agent

agents/recommendation_agent.py

In [ ]:
class RecommendationAgent:

    def recommend(
        self,
        user_preferences,
        options
    ):

        preferred = user_preferences.get(
            "category"
        )

        filtered = [

            x for x in options

            if x["category"] == preferred
        ]

        if filtered:
            return filtered[0]

        return options[0]

Enhanced Orchestrator

agents/orchestrator.py

In [ ]:
from agents.intent_agent import understand_intent
from agents.planner_agent import create_plan

from agents.memory_agent import MemoryAgent
from agents.self_healing_agent import SelfHealingAgent
from agents.analytics_agent import AnalyticsAgent

memory = MemoryAgent()
healer = SelfHealingAgent()
analytics = AnalyticsAgent()


def execute_task(task,user):

    try:

        intent = understand_intent(task)

        plan = create_plan(task)

        memory.save_preference(
            user,
            "last_task",
            task
        )

        return {

            "status":"success",
            "intent":intent,
            "plan":plan,
            "memory":
                memory.get_preferences(user)
        }

    except Exception as e:

        action = healer.recover(
            "timeout"
        )

        return {

            "status":"recovered",
            "action":action
        }